In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('superstore.csv', encoding='latin-1')

# Basic info
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'superstore.csv'

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

In [9]:
df.shape

(9994, 26)

In [10]:
df['Sales'].sum().round(2)

np.float64(2297200.86)

In [11]:
df['Profit'].sum().round(2)

np.float64(286397.02)

In [12]:
df['Order ID'].nunique()

5009

In [13]:
quarterly_trend = df.groupby(['Year','Quarter'])['Sales'].sum().reset_index()
quarterly_trend.columns = ['Year','Quarter','Quarterly_Revenue']
quarterly_trend['Quarterly_Revenue'] = quarterly_trend['Quarterly_Revenue'].round(2)
quarterly_trend

,Year,Quarter,Quarterly_Revenue
0,2014,1,74447.80
1,2014,2,86538.76
2,2014,3,143633.21
3,2014,4,179627.73
4,2015,1,68851.74
5,2015,2,89124.19
6,2015,3,130259.58
7,2015,4,182297.01
8,2016,1,93237.18
9,2016,2,136082.30


In [14]:
q4_avg = quarterly_trend[quarterly_trend['Quarter']==4]['Quarterly_Revenue'].mean()
other_avg = quarterly_trend[quarterly_trend['Quarter']!=4]['Quarterly_Revenue'].mean()
round(((q4_avg-other_avg)/other_avg)*100, 1)

np.float64(85.6)

In [5]:
df = pd.read_csv('superstore_cleaned.csv', encoding='latin-1')
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Quarter'] = df['Order Date'].dt.quarter
df['Profit Margin %'] = (df['Profit'] / df['Sales'] * 100).round(2)
print("Loaded! Shape:", df.shape)

Loaded! Shape: (9994, 26)


In [6]:
print("Total Rows:", df.shape[0])
print("Total Revenue:", round(df['Sales'].sum(), 2))
print("Total Profit:", round(df['Profit'].sum(), 2))
print("Total Orders:", df['Order ID'].nunique())
print("Years:", sorted(df['Year'].unique()))

Total Rows: 9994
Total Revenue: 2297200.86
Total Profit: 286397.02
Total Orders: 5009
Years: [np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017)]


In [7]:
quarterly_trend = df.groupby(['Year','Quarter'])['Sales'].sum().reset_index()
quarterly_trend.columns = ['Year','Quarter','Quarterly_Revenue']
quarterly_trend['Quarterly_Revenue'] = quarterly_trend['Quarterly_Revenue'].round(2)

best = quarterly_trend.loc[quarterly_trend['Quarterly_Revenue'].idxmax()]
worst = quarterly_trend.loc[quarterly_trend['Quarterly_Revenue'].idxmin()]
q4_avg = quarterly_trend[quarterly_trend['Quarter']==4]['Quarterly_Revenue'].mean()
other_avg = quarterly_trend[quarterly_trend['Quarter']!=4]['Quarterly_Revenue'].mean()

print("Best Quarter: Q{} {} → ${}".format(int(best['Quarter']), int(best['Year']), best['Quarterly_Revenue']))
print("Worst Quarter: Q{} {} → ${}".format(int(worst['Quarter']), int(worst['Year']), worst['Quarterly_Revenue']))
print("Q4 outperforms others by:", round(((q4_avg-other_avg)/other_avg)*100, 1), "%")

Best Quarter: Q4 2017 → $280054.07
Worst Quarter: Q1 2015 → $68851.74
Q4 outperforms others by: 85.6 %


In [ ]:
import pandasql as psql

# Install pandasql first
import subprocess
subprocess.run(['pip', 'install', 'pandasql'])

# ---- CLEANING ----
# Check missing values
print("Missing values:\n", df.isnull().sum())

# Fix date columns
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Extract useful time features
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Quarter'] = df['Order Date'].dt.quarter

# Add profit margin column
df['Profit Margin %'] = (df['Profit'] / df['Sales'] * 100).round(2)

# Add shipping days
df['Ship Days'] = (df['Ship Date'] - df['Order Date']).dt.days

print("\nCleaning done! Shape:", df.shape)
print("\nNew columns added:", ['Year', 'Month', 'Quarter', 'Profit Margin %', 'Ship Days'])
df.head()

In [ ]:
!pip install pandasql


In [ ]:
import pandasql as psql

# Install pandasql first
import subprocess
subprocess.run(['pip', 'install', 'pandasql'])

# ---- CLEANING ----
# Check missing values
print("Missing values:\n", df.isnull().sum())

# Fix date columns
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Extract useful time features
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Quarter'] = df['Order Date'].dt.quarter

# Add profit margin column
df['Profit Margin %'] = (df['Profit'] / df['Sales'] * 100).round(2)

# Add shipping days
df['Ship Days'] = (df['Ship Date'] - df['Order Date']).dt.days

print("\nCleaning done! Shape:", df.shape)
print("\nNew columns added:", ['Year', 'Month', 'Quarter', 'Profit Margin %', 'Ship Days'])
df.head()

In [ ]:
# ---- SQL QUERIES ----

# Query 1 — Revenue and Profit by Region
query1 = """
SELECT
    Region,
    ROUND(SUM(Sales), 2) AS Total_Revenue,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(SUM(Profit)/SUM(Sales)*100, 2) AS Profit_Margin_Pct,
    COUNT(DISTINCT "Order ID") AS Total_Orders
FROM df
GROUP BY Region
ORDER BY Total_Revenue DESC
"""
region_summary = psql.sqldf(query1)
print("=== Revenue & Profit by Region ===")
print(region_summary)

# Query 2 — Revenue by Category
query2 = """
SELECT
    Category,
    ROUND(SUM(Sales), 2) AS Total_Revenue,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(SUM(Profit)/SUM(Sales)*100, 2) AS Profit_Margin_Pct
FROM df
GROUP BY Category
ORDER BY Total_Revenue DESC
"""
category_summary = psql.sqldf(query2)
print("\n=== Revenue & Profit by Category ===")
print(category_summary)

# Query 3 — Top 10 products by Revenue
query3 = """
SELECT
    "Product Name",
    Category,
    ROUND(SUM(Sales), 2) AS Total_Revenue,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM df
GROUP BY "Product Name", Category
ORDER BY Total_Revenue DESC
LIMIT 10
"""
top_products = psql.sqldf(query3)
print("\n=== Top 10 Products by Revenue ===")
print(top_products)

# Query 4 — Bottom 5 products by Profit Margin
query4 = """
SELECT
    "Product Name",
    Category,
    ROUND(SUM(Sales), 2) AS Total_Revenue,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(SUM(Profit)/SUM(Sales)*100, 2) AS Profit_Margin_Pct
FROM df
GROUP BY "Product Name", Category
HAVING SUM(Sales) > 1000
ORDER BY Profit_Margin_Pct ASC
LIMIT 5
"""
bottom_products = psql.sqldf(query4)
print("\n=== Bottom 5 Products by Profit Margin ===")
print(bottom_products)

# Query 5 - Quarterly Revenue trend
query5 = """
SELECT
    Year,
    Quarter,
    ROUND(SUM(Sales), 2) AS Quarterly_Revenue
FROM df
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
quarterly_trend = psql.sqldf(query5)
print("\n=== Quarterly Revenue Trend ===")
print(quarterly_trend)

In [ ]:
# ---- EDA VISUALIZATIONS ----

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
fig.suptitle('Retail Sales Performance Analysis', fontsize=20, fontweight='bold', y=0.98)

# ---- Chart 1: Revenue by Region ----
regions = ['West', 'East', 'Central', 'South']
revenues = [725457.82, 678781.24, 501239.89, 391721.91]
colors1 = ['#1B3A6B', '#378ADD', '#534AB7', '#93B4D8']

bars1 = axes[0,0].bar(regions, revenues, color=colors1, edgecolor='white', linewidth=0.5)
axes[0,0].set_title('Total Revenue by Region', fontweight='bold', fontsize=13)
axes[0,0].set_ylabel('Revenue ($)')
axes[0,0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0,0].set_facecolor('#F8F9FA')
for bar, val in zip(bars1, revenues):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                   f'${val:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# ---- Chart 2: Revenue vs Profit by Category ----
categories = ['Technology', 'Furniture', 'Office Supplies']
cat_revenue = [836154.03, 741999.80, 719047.03]
cat_profit = [145454.95, 18451.27, 122490.80]
x = np.arange(len(categories))
width = 0.35

bars_r = axes[0,1].bar(x - width/2, cat_revenue, width, label='Revenue', color='#1B3A6B', edgecolor='white')
bars_p = axes[0,1].bar(x + width/2, cat_profit, width, label='Profit', color='#EF9F27', edgecolor='white')
axes[0,1].set_title('Revenue vs Profit by Category', fontweight='bold', fontsize=13)
axes[0,1].set_xticks(x)
axes[0,1].set_xticklabels(categories)
axes[0,1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0,1].legend()
axes[0,1].set_facecolor('#F8F9FA')

# ---- Chart 3: Quarterly Revenue Trend ----
quarters_label = [f"Q{q}\n{y}" for y, q in zip(quarterly_trend['Year'], quarterly_trend['Quarter'])]
axes[1,0].plot(quarters_label, quarterly_trend['Quarterly_Revenue'],
               marker='o', color='#1B3A6B', linewidth=2.5, markersize=6)
axes[1,0].fill_between(quarters_label, quarterly_trend['Quarterly_Revenue'],
                        alpha=0.15, color='#1B3A6B')
axes[1,0].set_title('Quarterly Revenue Trend (2014–2017)', fontweight='bold', fontsize=13)
axes[1,0].set_ylabel('Revenue ($)')
axes[1,0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1,0].tick_params(axis='x', rotation=45, labelsize=7.5)
axes[1,0].set_facecolor('#F8F9FA')

# ---- Chart 4: Profit Margin by Region ----
margins = [14.94, 13.48, 7.92, 11.93]
colors4 = ['#1D9E75' if m > 10 else '#D85A30' for m in margins]
bars4 = axes[1,1].bar(regions, margins, color=colors4, edgecolor='white')
axes[1,1].set_title('Profit Margin % by Region', fontweight='bold', fontsize=13)
axes[1,1].set_ylabel('Profit Margin (%)')
axes[1,1].axhline(y=10, color='gray', linestyle='--', linewidth=1, label='10% benchmark')
axes[1,1].legend()
axes[1,1].set_facecolor('#F8F9FA')
for bar, val in zip(bars4, margins):
    axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                   f'{val}%', ha='center', fontsize=10, fontweight='bold')

# ---- Chart 5: Top 5 Products by Revenue ----
top5_names = ['Canon imageCLASS\n2200 Copier', 'Fellowes PB500\nBinding',
              'Cisco TelePresence\nEX90', 'HON 5400\nTask Chairs', 'GBC DocuBind\nTL300']
top5_rev = [61599.82, 27453.38, 22638.48, 21870.58, 19823.48]
colors5 = ['#1B3A6B','#378ADD','#534AB7','#93B4D8','#1D9E75']
bars5 = axes[2,0].barh(top5_names[::-1], top5_rev[::-1], color=colors5[::-1], edgecolor='white')
axes[2,0].set_title('Top 5 Products by Revenue', fontweight='bold', fontsize=13)
axes[2,0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[2,0].set_facecolor('#F8F9FA')
for bar, val in zip(bars5, top5_rev[::-1]):
    axes[2,0].text(bar.get_width() + 300, bar.get_y() + bar.get_height()/2,
                   f'${val:,.0f}', va='center', fontsize=9, fontweight='bold')

# ---- Chart 6: Bottom 5 by Profit Margin ----
bot5_names = ['Epson TM-T88V\nPrinter', 'Cubify CubeX\n3D (Double)',
              'BoxOffice\nConference Table', 'Bevis Conference\nTables', 'Cubify CubeX\n3D (Triple)']
bot5_margins = [-87.18, -80.00, -67.31, -58.26, -48.00]
bars6 = axes[2,1].barh(bot5_names[::-1], bot5_margins[::-1], color='#D85A30', edgecolor='white')
axes[2,1].set_title('Bottom 5 Products by Profit Margin', fontweight='bold', fontsize=13)
axes[2,1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
axes[2,1].axvline(x=0, color='black', linewidth=0.8)
axes[2,1].set_facecolor('#F8F9FA')
for bar, val in zip(bars6, bot5_margins[::-1]):
    axes[2,1].text(bar.get_width() - 2, bar.get_y() + bar.get_height()/2,
                   f'{val}%', va='center', ha='right', fontsize=9, fontweight='bold', color='white')

plt.tight_layout()
plt.savefig('eda_visualizations.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved as eda_visualizations.png")

In [ ]:
# Export all summary tables to CSV files
# We'll import these into Google Sheets

# 1. Region Summary
region_summary.to_csv('region_summary.csv', index=False)

# 2. Category Summary
category_summary.to_csv('category_summary.csv', index=False)

# 3. Top 10 Products
top_products.to_csv('top_products.csv', index=False)

# 4. Bottom 5 Products
bottom_products.to_csv('bottom_products.csv', index=False)

# 5. Quarterly Trend
quarterly_trend.to_csv('quarterly_trend.csv', index=False)

# 6. Full cleaned dataset
df.to_csv('superstore_cleaned.csv', index=False)

print("All files exported successfully!")
print("Files ready to download:")
print("- region_summary.csv")
print("- category_summary.csv")
print("- top_products.csv")
print("- bottom_products.csv")
print("- quarterly_trend.csv")
print("- superstore_cleaned.csv")

In [ ]:
print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])
print("Years in data:", sorted(df['Year'].unique()))
print("Total Revenue:", round(df['Sales'].sum(), 2))
print("Total Profit:", round(df['Profit'].sum(), 2))
print("Total Orders:", df['Order ID'].nunique())